# IPN 13 lớp: baseline skeleton chạy lại trên dữ liệu của mình

Mục tiêu: có bảng so sánh **công bằng** giữa mô hình Transformer (neural C) và các baseline skeleton đơn giản trong các paper IPN gần đây, **cùng dữ liệu, cùng split, cùng đầu vào 52 keypoint, cùng cách chọn checkpoint**, 3 seed, chỉ validation.

| Baseline | Cấu hình | Tương ứng paper |
|---|---|---|
| `lstm_paper` | LSTM 3 tầng, hidden 256, dropout 0,3, lr 1e-3, batch 64, 30 epoch, CE chuẩn, không augmentation | IPN HandS (2025) |
| `lstm_c` | LSTM 4 tầng, hidden 192, **recipe C** (label smoothing, class weight, jitter, dropout keypoint, patience 12) | nhóm RNN, cùng recipe với Transformer |
| `bigru_c` | BiGRU, recipe C | nhóm RNN (MediaPipe + RNN, 2025) |
| `tcn_c` | TCN, recipe C | baseline tích chập thời gian |
| `mlp_mixer_c` | MLP-Mixer, recipe C | baseline MLP |
| `c_ref` | Transformer neural C (dùng lại run có sẵn trên Drive) | mô hình của mình |

Khác paper: đầu vào là 52 keypoint RTMW (không phải 21 điểm một tay), 64 frame resample, 13 lớp, split và dữ liệu của mình. Vì vậy **không so số tuyệt đối với paper**; chỉ so các dòng trong bảng này với nhau. Chỉ báo cáo validation; test giữ khóa cho lần so sánh cuối. Có báo cáo cả average recall (số mà paper HandS dùng).

In [ ]:
from pathlib import Path
import csv
import datetime as dt
import hashlib
import json
import os
import shutil
import statistics
import subprocess
import sys
import uuid
import zipfile

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Không chạy trên Colab: bỏ qua mount Drive.')

# --- Chỉ sửa các biến này ---
DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/ipn-extract-data')
REPO_URL = 'https://github.com/HuyenHoang28/DynamicHandGestureWithXAI.git'
REPO_REF = 'HuyenHoang'      # nhánh chứa code mới (hand_geometry.py, ensemble_neural_only.py)
REPO_COMMIT = None           # đặt hash để ghim chính xác; None thì dùng đầu nhánh và ghi lại hash đã dùng
SEEDS = [42, 123, 3407]
RUN_LOCKED_TEST = False      # test giữ khóa; baseline chỉ so trên validation
# Baseline cần chạy. Bỏ bớt tên để giảm thời gian (mỗi baseline = 3 run).
BASELINES = ['lstm_paper', 'lstm_c', 'bigru_c', 'tcn_c', 'mlp_mixer_c']
NUM_CLASSES = 13
EXPECTED_COUNTS = {'train': 2443, 'val': 589, 'test': 1101}   # None để bỏ kiểm tra khi chạy thử nhỏ
DEVICE = 'cuda:0'

RUN_ID = dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:10]
WORK_ROOT = Path('/content') / f'ipn13_skeleton_baselines_{RUN_ID}'
REPO_DIR = WORK_ROOT / 'repo'
OUTPUT_ROOT = DRIVE_DATA_ROOT / 'outputs' / 'ipn13_neural_sweep'        # checkpoint từng run, giữ qua reconnect
DRIVE_RUN_ROOT = DRIVE_DATA_ROOT / 'runs' / 'ipn13_skeleton_baselines'
TRAIN_FILE = DRIVE_DATA_ROOT / 'train.txt'
VAL_FILE = DRIVE_DATA_ROOT / 'val.txt'
TEST_FILE = DRIVE_DATA_ROOT / 'test.txt'
print({'run_id': RUN_ID, 'seeds': SEEDS, 'locked_test': RUN_LOCKED_TEST})


## 1. Clone code (phải push nhánh trước)

In [ ]:
COMMANDS = []
WORK_ROOT.mkdir(parents=True, exist_ok=True)


def run_argv(argv, cwd=None, log_path=None):
    command = [str(value) for value in argv]
    COMMANDS.append(command)
    print('$', ' '.join(command))
    log = open(log_path, 'a', encoding='utf-8') if log_path else None
    try:
        process = subprocess.Popen(command, cwd=str(cwd) if cwd else None, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            if log:
                log.write(line)
        if process.wait() != 0:
            raise subprocess.CalledProcessError(process.returncode, command)
    finally:
        if log:
            log.close()


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


if not REPO_DIR.exists():
    run_argv(['git', 'clone', '--filter=blob:none', REPO_URL, REPO_DIR])
run_argv(['git', '-C', REPO_DIR, 'fetch', 'origin', REPO_COMMIT or REPO_REF])
run_argv(['git', '-C', REPO_DIR, 'checkout', '--detach', REPO_COMMIT or 'FETCH_HEAD'])
REPO_COMMIT_RESOLVED = subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], check=True, text=True,
                                      capture_output=True).stdout.strip()
if REPO_COMMIT:
    assert REPO_COMMIT_RESOLVED == REPO_COMMIT, (REPO_COMMIT_RESOLVED, REPO_COMMIT)
NEURAL_DIR = REPO_DIR / 'final' / 'code' / 'neural_only_experiments'
for required in ('train_neural_only.py', 'test_neural_only.py', 'ensemble_neural_only.py'):
    assert (NEURAL_DIR / required).is_file(), f'{required} chưa có trên nhánh {REPO_REF}: hãy commit và push code mới trước.'
if (REPO_DIR / 'requirements.txt').is_file():
    run_argv([sys.executable, '-m', 'pip', 'install', '-q', '-r', REPO_DIR / 'requirements.txt'])
print('Repository revision:', REPO_COMMIT_RESOLVED)

assert 'backbone == "lstm"' in (NEURAL_DIR / 'models.py').read_text(encoding='utf-8'), 'Backbone lstm chưa có trên nhánh: push code mới.'


## 2. Kiểm tra dữ liệu trên Drive

In [ ]:
def read_manifest(path):
    rows = []
    for line in Path(path).read_text(encoding='utf-8').splitlines():
        if line.strip():
            source, _, label = line.rsplit(maxsplit=2)
            rows.append((source, int(label)))
    return rows


manifests = {name: read_manifest(path) for name, path in (('train', TRAIN_FILE), ('val', VAL_FILE), ('test', TEST_FILE))}
for name, rows in manifests.items():
    ids = {label for _, label in rows}
    if EXPECTED_COUNTS:
        assert len(rows) == EXPECTED_COUNTS[name], (name, len(rows))
    assert ids == set(range(NUM_CLASSES)), (name, sorted(ids))
sources = {name: {source.replace(chr(92), '/') for source, _ in rows} for name, rows in manifests.items()}
assert not (sources['train'] & sources['val']) and not (sources['train'] & sources['test']) and not (sources['val'] & sources['test'])

CACHE_ROOT = None
for candidate in (DRIVE_DATA_ROOT / 'keypoint_tensor_cache', DRIVE_DATA_ROOT / 'data' / 'keypoint_tensor_cache'):
    if candidate.is_dir():
        CACHE_ROOT = candidate
        break
assert CACHE_ROOT is not None, 'Không thấy keypoint_tensor_cache trên Drive.'
print({'counts': {name: len(rows) for name, rows in manifests.items()}, 'cache_root': str(CACHE_ROOT)})


## 3. Huấn luyện (chỉ khi thiếu run)

Run hoàn tất được dùng lại; run dở dang bị đổi tên, không xóa.

In [ ]:
BASE_ARGS = [
    '--backbone', 'transformer', '--num-classes', NUM_CLASSES, '--num-frames', 64, '--batch-size', 64,
    '--epochs', 80, '--d-model', 192, '--layers', 4, '--heads', 8, '--dropout', 0.15,
    '--label-smoothing', 0.05, '--pooling', 'masked_mean', '--class-weight-power', 0.5,
    '--jitter-std', 0.005, '--frame-mask-probability', 0.0, '--keypoint-dropout-probability', 0.02,
    '--select-metric', 'macro_f1', '--early-stop-patience', 12, '--num-workers', 0, '--log-every', 20,
]
# Tham số đặt sau BASE_ARGS ghi đè giá trị trước đó (argparse lấy lần xuất hiện cuối).
ALL_VARIANTS = {
    'lstm_paper': ['--backbone', 'lstm', '--d-model', 256, '--layers', 3, '--dropout', 0.3, '--lr', 0.001,
                   '--batch-size', 64, '--epochs', 30, '--early-stop-patience', 30, '--label-smoothing', 0,
                   '--class-weight-power', 0, '--jitter-std', 0, '--keypoint-dropout-probability', 0],
    'lstm_c': ['--backbone', 'lstm'],
    'bigru_c': ['--backbone', 'bigru'],
    'tcn_c': ['--backbone', 'tcn'],
    'mlp_mixer_c': ['--backbone', 'mlp_mixer'],
}
VARIANTS = {'c_ref': []}
VARIANTS.update({name: ALL_VARIANTS[name] for name in BASELINES})
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIRS = {}
for variant, extra in VARIANTS.items():
    for seed in SEEDS:
        run_dir = OUTPUT_ROOT / f'{variant}_seed{seed}'
        RUN_DIRS[(variant, seed)] = run_dir
        marker = run_dir / 'SWEEP_RUN_COMPLETE.json'
        if marker.is_file() and (run_dir / 'best.pt').is_file():
            print('Dùng lại run đã hoàn tất:', run_dir.name)
            continue
        if run_dir.exists() and any(run_dir.iterdir()):
            stale = run_dir.with_name(run_dir.name + '_incomplete_' + dt.datetime.now().strftime('%Y%m%dT%H%M%S'))
            run_dir.rename(stale)
            print('Run dở dang được đổi tên thành', stale.name)
        command = [sys.executable, '-u', NEURAL_DIR / 'train_neural_only.py',
                   '--cache-root', CACHE_ROOT, '--train-file', TRAIN_FILE, '--val-file', VAL_FILE,
                   '--output-dir', run_dir, '--seed', seed, '--device', DEVICE] + BASE_ARGS + extra
        assert '--test-file' not in command
        run_dir.mkdir(parents=True)
        run_argv(command, cwd=NEURAL_DIR, log_path=run_dir / 'train.log')
        assert (run_dir / 'best.pt').is_file()
        marker.write_text(json.dumps({'variant': variant, 'seed': seed, 'commit': REPO_COMMIT_RESOLVED,
                                      'command': [str(v) for v in command]}, indent=2), encoding='utf-8')
print('Xong huấn luyện:', len(RUN_DIRS), 'run')


## 4. Đánh giá validation: từng seed và ensemble

In [ ]:
def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


val_metrics = {}
for (variant, seed), run_dir in RUN_DIRS.items():
    metrics_path = run_dir / 'val_eval' / 'metrics.json'
    if not metrics_path.is_file():
        run_argv([sys.executable, '-u', NEURAL_DIR / 'test_neural_only.py', '--checkpoint', run_dir / 'best.pt',
                  '--cache-root', CACHE_ROOT, '--test-file', VAL_FILE, '--split', 'val', '--batch-size', 128,
                  '--num-workers', 0, '--device', DEVICE, '--timing-warmup-batches', 1, '--timing-repeats', 1,
                  '--output', metrics_path], cwd=NEURAL_DIR)
    val_metrics[(variant, seed)] = read_json(metrics_path)

ENSEMBLE_ROOT = OUTPUT_ROOT / 'ensembles'
ensemble_val = {}
for variant in VARIANTS:
    out = ENSEMBLE_ROOT / variant / 'val' / 'metrics.json'
    if not out.is_file():
        run_argv([sys.executable, '-u', NEURAL_DIR / 'ensemble_neural_only.py', '--checkpoints',
                  *[RUN_DIRS[(variant, seed)] / 'best.pt' for seed in SEEDS],
                  '--cache-root', CACHE_ROOT, '--test-file', VAL_FILE, '--split', 'val', '--device', DEVICE,
                  '--output', out], cwd=NEURAL_DIR)
    ensemble_val[variant] = read_json(out)
print('Đánh giá validation xong.')


## 5. Bảng baseline

In [ ]:
def stats(values):
    return statistics.mean(values), (statistics.stdev(values) if len(values) > 1 else 0.0)


def params_of(variant):
    import torch
    return sum(v.numel() for v in torch.load(RUN_DIRS[(variant, SEEDS[0])] / 'best.pt', map_location='cpu',
                                             weights_only=False)['model'].values())


summary = []
print(f"{'model':12} {'params':>9} {'acc':>15} {'macroF1':>15} {'avg recall':>15} {'ens acc':>8} {'ens F1':>8}")
for variant in VARIANTS:
    accs = [val_metrics[(variant, s)]['accuracy'] for s in SEEDS]
    f1s = [val_metrics[(variant, s)]['macro_f1'] for s in SEEDS]
    recs = [val_metrics[(variant, s)]['macro_recall'] for s in SEEDS]
    ens = ensemble_val[variant]
    row = {'model': variant, 'params': params_of(variant),
           'acc_mean': stats(accs)[0], 'acc_std': stats(accs)[1],
           'f1_mean': stats(f1s)[0], 'f1_std': stats(f1s)[1],
           'recall_mean': stats(recs)[0], 'recall_std': stats(recs)[1],
           'ensemble_acc': ens['accuracy'], 'ensemble_f1': ens['macro_f1']}
    summary.append(row)
    print(f"{variant:12} {row['params']:9,d} {row['acc_mean']:.4f}±{row['acc_std']:.4f} {row['f1_mean']:.4f}±{row['f1_std']:.4f} "
          f"{row['recall_mean']:.4f}±{row['recall_std']:.4f} {row['ensemble_acc']:8.4f} {row['ensemble_f1']:8.4f}")

print('\nHiệu macro-F1 theo cặp seed (baseline - c_ref):')
verdicts = {}
for variant in VARIANTS:
    if variant == 'c_ref':
        continue
    deltas = [val_metrics[(variant, s)]['macro_f1'] - val_metrics[('c_ref', s)]['macro_f1'] for s in SEEDS]
    verdicts[variant] = {'deltas': deltas, 'mean': statistics.mean(deltas)}
    print(f"  {variant:12} " + '  '.join(f'{d:+.4f}' for d in deltas) + f"   trung bình {statistics.mean(deltas):+.4f}")
    if all(d > 0 for d in deltas):
        print('    -> baseline TỐT HƠN c_ref ở cả 3 seed')
    elif all(d < 0 for d in deltas):
        print('    -> baseline kém hơn c_ref ở cả 3 seed')
    else:
        print('    -> lẫn lộn giữa các seed')


## 6. Các lớp khó

In [ ]:
names = [row['label_name'] for row in val_metrics[('c_ref', SEEDS[0])]['per_class']]
print('F1 trung bình qua seed (validation) các lớp khó:')
print(f"{'class':6}" + ''.join(f'{v:>13}' for v in VARIANTS))
for index, name in enumerate(names):
    if name in ['G01', 'G02', 'G08', 'G09', 'G10', 'G11']:
        print(f'{name:6}' + ''.join(
            f"{statistics.mean(val_metrics[(v, s)]['per_class'][index]['f1'] for s in SEEDS):13.3f}" for v in VARIANTS))


## 7. Đóng gói ZIP (từ chối ghi đè)

In [ ]:
DRIVE_RUN_ROOT.mkdir(parents=True, exist_ok=True)
run_dir_drive = DRIVE_RUN_ROOT / RUN_ID
run_zip = DRIVE_RUN_ROOT / f'{RUN_ID}.zip'
if run_dir_drive.exists() or run_zip.exists():
    raise FileExistsError(f'Từ chối ghi đè: {run_dir_drive} hoặc {run_zip}')
payload = run_dir_drive / 'payload'
payload.mkdir(parents=True)
(payload / 'summary_validation.json').write_text(json.dumps({'summary': summary, 'paired_deltas': verdicts,
                                                               'commit': REPO_COMMIT_RESOLVED}, indent=2), encoding='utf-8')
(payload / 'commands.json').write_text(json.dumps(COMMANDS, indent=2), encoding='utf-8')
for variant in VARIANTS:
    target = payload / 'validation' / variant
    target.mkdir(parents=True)
    shutil.copy2(ENSEMBLE_ROOT / variant / 'val' / 'metrics.json', target / 'ensemble_val_metrics.json')
    for seed in SEEDS:
        shutil.copy2(RUN_DIRS[(variant, seed)] / 'val_eval' / 'metrics.json', target / f'seed{seed}_val_metrics.json')
        shutil.copy2(RUN_DIRS[(variant, seed)] / 'history.json', target / f'seed{seed}_history.json')
with zipfile.ZipFile(run_zip, 'x', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(run_dir_drive.rglob('*')):
        if path.is_file():
            archive.write(path, arcname=path.relative_to(run_dir_drive))
print('Run ZIP:', run_zip)
